# Data Type Conversion

## Definition
When you load data from CSV or databases, pandas often infers incorrect types (e.g., reading numbers as strings).
Correct data types are essential because:
- Operations behave differently per dtype (string `'10' + '20' = '1020'`, not 30)
- Memory usage depends on dtype (object >> int8)
- ML algorithms require numeric inputs
- Datetime operations require proper datetime dtype

### Common Conversions
| From | To | Method |
|------|----|--------|
| object (str) | int/float | `pd.to_numeric()` |
| object | datetime | `pd.to_datetime()` |
| object | category | `.astype('category')` |
| int64 | int8/int16 | `.astype(np.int8)` |
| float64 | float32 | `.astype(np.float32)` |
| bool | int | `.astype(int)` |

In [ ]:
import pandas as pd
import numpy as np

# Simulate raw CSV data (everything loaded as string/object)
df = pd.DataFrame({
    'id':       ['1', '2', '3', '4', '5'],
    'age':      ['28', '34', '25', 'N/A', '29'],
    'salary':   ['75000.0', '85000', '', '95000.0', '70000.0'],
    'joined':   ['2020-01-15', '2019-03-22', '2021-07-01', '2018-11-30', '2022-02-14'],
    'dept':     ['Eng', 'Mkt', 'Eng', 'HR', 'Eng'],
    'is_active': ['True', 'False', 'True', 'True', 'False']
})
print(df)
print()
print(df.dtypes)

In [ ]:
# pd.to_numeric() — robust numeric conversion
df['id'] = pd.to_numeric(df['id'])
df['age'] = pd.to_numeric(df['age'], errors='coerce')   # 'N/A' -> NaN
df['salary'] = pd.to_numeric(df['salary'], errors='coerce')  # '' -> NaN
print('After numeric conversion:')
print(df[['id', 'age', 'salary']])
print(df[['id', 'age', 'salary']].dtypes)

In [ ]:
# pd.to_datetime() — date parsing
df['joined'] = pd.to_datetime(df['joined'])
print('After datetime conversion:')
print(df['joined'])
print('dtype:', df['joined'].dtype)

In [ ]:
# astype() — direct type casting
df['dept'] = df['dept'].astype('category')
df['is_active'] = df['is_active'].map({'True': True, 'False': False}).astype(bool)

print('After astype:')
print(df.dtypes)

In [ ]:
# Memory optimization via dtype downcast
print('Memory before optimization:')
df_num = pd.DataFrame({
    'a': np.random.randint(0, 100, 10000).astype(np.int64),
    'b': np.random.randn(10000).astype(np.float64)
})
print(df_num.memory_usage(deep=True).sum() / 1024, 'KB')

# Downcast
df_num['a'] = df_num['a'].astype(np.int8)    # 0-100 fits in int8!
df_num['b'] = df_num['b'].astype(np.float32)

print('Memory after optimization:')
print(df_num.memory_usage(deep=True).sum() / 1024, 'KB')

In [ ]:
# pd.to_numeric with downcast
s = pd.Series([1.0, 2.0, 3.0, 4.0])
print('Original dtype:', s.dtype)

s_int = pd.to_numeric(s, downcast='integer')
print('Downcasted integer:', s_int.dtype)

s_float = pd.to_numeric(s, downcast='float')
print('Downcasted float:', s_float.dtype)

In [ ]:
# EXCEPTION: Cannot cast invalid values
try:
    pd.to_numeric(pd.Series(['1', '2', 'abc']), errors='raise')
except ValueError as e:
    print('ValueError:', e)

# Use errors='coerce' to handle gracefully
s = pd.to_numeric(pd.Series(['1', '2', 'abc']), errors='coerce')
print('errors=coerce:', s.values)  # 'abc' -> NaN